# Lamio v2 - MoE Inference (Ornith-35B) on Kaggle T4

Auto-detects GPU (CUDA/Metal/Vulkan), builds ggml + Lamio v2, downloads model to /kaggle/working (persistent), runs inference with ExpertRouter.


In [ ]:
#@title System Check
import os, subprocess, psutil

print(f'CPU cores: {os.cpu_count()}')
print(f'RAM: {psutil.virtual_memory().total / 1024**3:.1f} GB')
print(f'RAM available: {psutil.virtual_memory().available / 1024**3:.1f} GB')
print(f'Disk /kaggle/working free: {psutil.disk_usage("/kaggle/working").free / 1024**3:.1f} GB')

import torch
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB)')
else:
    print('No GPU detected')

os.makedirs('/kaggle/working/models', exist_ok=True)
os.makedirs('/kaggle/working/lamio', exist_ok=True)
print('\nPersistent dirs ready: /kaggle/working/models, /kaggle/working/lamio')


In [ ]:
#@title Clone Lamio + Build ggml (CUDA) + Lamio v2
import subprocess, os, time, glob, shutil

REPO = 'https://github.com/Neguiolidas/Lamio.git'
PERSIST = '/kaggle/working/lamio'
LAMIO_DIR = '/tmp/lamio'

# Use cached build if exists (saves 25min on re-runs)
if os.path.exists(f'{PERSIST}/v2/build/src/lamio'):
    print(f'Using cached build at {PERSIST}')
    if os.path.exists(LAMIO_DIR):
        shutil.rmtree(LAMIO_DIR)
    subprocess.run(['cp', '-a', PERSIST, LAMIO_DIR], check=True)
    subprocess.run(['git', '-C', LAMIO_DIR, 'pull'], capture_output=True)
else:
    # Fresh clone + build
    if os.path.exists(LAMIO_DIR):
        shutil.rmtree(LAMIO_DIR)
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main', REPO, LAMIO_DIR], check=True)

assert os.path.exists(f'{LAMIO_DIR}/v2/CMakeLists.txt'), 'v2/CMakeLists.txt not found!'
print(f'Repo at {LAMIO_DIR}')
subprocess.run(['git', '-C', LAMIO_DIR, 'log', '--oneline', '-3'], check=True)

# Build ggml with CUDA (NO_VMM avoids libcuda.so link requirement on Kaggle)
print('\n--- Building ggml (CUDA) ---')
build_dir = f'{LAMIO_DIR}/build'
os.makedirs(build_dir, exist_ok=True)
result = subprocess.run([
    'cmake', '-B', build_dir, '-S', LAMIO_DIR,
    '-DCMAKE_BUILD_TYPE=Release',
    '-DGGML_CUDA=ON',
    '-DGGML_CUDA_NO_VMM=ON',
    '-DGGML_NATIVE=ON', '-DGGML_AVX2=ON',
    '-DGGML_FMA=ON', '-DGGML_F16C=ON', '-DGGML_AVX=ON',
    '-DLLAMA_CURL=OFF', '-DBUILD_SHARED_LIBS=ON',
    '-DGGML_CUDA_FORCE_MMQ=OFF',
], capture_output=True, text=True)
print(result.stdout[-400:] if result.stdout else '')
if result.returncode != 0:
    print('CMAKE ERROR:', result.stderr[-1000:])
    raise SystemExit(1)

t0 = time.time()
result = subprocess.run(['cmake', '--build', build_dir, '-j', '4', '--target', 'ggml-cpu'],
    capture_output=True, text=True, timeout=300)
print(f'ggml-cpu: {time.time()-t0:.0f}s exit={result.returncode}')

print('Building ggml-cuda (10-20 min on first run)...')
t0 = time.time()
result = subprocess.run(['cmake', '--build', build_dir, '-j', '2', '--target', 'ggml-cuda'],
    capture_output=True, text=True, timeout=1500)
print(f'ggml-cuda: {time.time()-t0:.0f}s exit={result.returncode}')
if result.returncode != 0:
    print('CUDA BUILD ERROR:', result.stderr[-1000:])

result = subprocess.run(['cmake', '--build', build_dir, '-j', '4', '--target', 'ggml'],
    capture_output=True, text=True, timeout=120)
print(f'ggml base: exit={result.returncode}')

libs = glob.glob(f'{build_dir}/bin/libggml*')
print('libs:', [os.path.basename(x) for x in libs])

# Build Lamio v2
print('\n--- Building Lamio v2 ---')
v2_build = f'{LAMIO_DIR}/v2/build'
os.makedirs(v2_build, exist_ok=True)
result = subprocess.run([
    'cmake', '-B', v2_build, '-S', f'{LAMIO_DIR}/v2',
    f'-DGGML_BUILD_DIR={build_dir}',
    f'-DGGML_ROOT={LAMIO_DIR}/ggml',
], capture_output=True, text=True)
if result.returncode != 0:
    print('CMAKE ERR:', result.stderr[-500:])

result = subprocess.run(['cmake', '--build', v2_build, '-j', '4'],
    capture_output=True, text=True, timeout=180)
print(f'Lamio v2: exit={result.returncode}')
if result.returncode != 0:
    print('LAMIO ERR:', result.stderr[-1000:])

lamio_bin = f'{v2_build}/src/lamio'
if os.path.exists(lamio_bin):
    os.chmod(lamio_bin, 0o755)
    # Cache to persistent storage for re-runs
    subprocess.run(['cp', '-a', LAMIO_DIR, PERSIST], check=True)
    print(f'\nBuild cached to {PERSIST}')
    print(f'OK: {lamio_bin}')
else:
    print(f'ERROR: {lamio_bin} not found!')


In [ ]:
#@title Download Ornith-35B MXFP4 MoE to /kaggle/working/models (persistent)
import os, time
from kaggle_secrets import UserSecretsClient
from huggingface_hub import hf_hub_download

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

REPO_ID = 'jashepp/Ornith-1.0-35B-A3B-MXFP4_MOE_Hybrid-Imatrix-GGUF'
FILENAME = 'Ornith-1.0-35B-MXFP4_MOE-Only-Imatrix.gguf'
MODEL_DIR = '/kaggle/working/models'
TARGET = f'{MODEL_DIR}/{FILENAME}'

os.makedirs(MODEL_DIR, exist_ok=True)

if os.path.exists(TARGET) and os.path.getsize(TARGET) > 17 * 1024**3:
    print(f'Model cached: {os.path.getsize(TARGET) / 1024**3:.1f} GB')
else:
    print(f'Downloading {FILENAME} from {REPO_ID}')
    t0 = time.time()
    downloaded = hf_hub_download(
        repo_id=REPO_ID,
        filename=FILENAME,
        local_dir=MODEL_DIR,
        token=hf_token,
        local_dir_use_symlinks=False,
    )
    elapsed = time.time() - t0
    size_gb = os.path.getsize(downloaded) / 1024**3
    print(f'\nDownloaded: {size_gb:.1f} GB in {elapsed:.0f}s ({size_gb/elapsed:.2f} GB/s)')

print(f'Model: {TARGET} ({os.path.getsize(TARGET) / 1024**3:.1f} GB)')


In [ ]:
#@title Run Inference (ChatML, GPU auto-detect, ExpertRouter)
import subprocess, os, time, psutil

LAMIO_BIN = '/tmp/lamio/v2/build/src/lamio'
MODEL_PATH = '/kaggle/working/models/Ornith-1.0-35B-MXFP4_MOE-Only-Imatrix.gguf'

ENV = dict(os.environ,
    LD_LIBRARY_PATH='/tmp/lamio/build/bin',
    LAMIO_MAX_RSS_MB='4096',
    LAMIO_THREADS=str(os.cpu_count()),
)

PROMPT = 'how many is 50+50?'

# Build plain ChatML prompt (no Jinja2, no thinking injection)
CHATML = f'<|im_start|>user\n{PROMPT}<|im_end|>\n<|im_start|>assistant\n'

cmd = [LAMIO_BIN, MODEL_PATH,
       '--generate', '--prompt', CHATML,
       '--n-gen', '128', '--temp', '0.3',
       '--threads', str(os.cpu_count()),
       '--max-rss-mb', '4096',
       '--auto-stop']

print(f'Running: {os.path.basename(LAMIO_BIN)} {os.path.basename(MODEL_PATH)} ...')
print(f'Prompt: {repr(CHATML)}')
print(f'CPU threads: {os.cpu_count()}')
print(f'\n=== RAM BEFORE ===')
print(f'  Available: {psutil.virtual_memory().available / 1024**3:.1f} GB')

# Check GPU before
try:
    import torch
    if torch.cuda.is_available():
        print(f'  GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB)')
except: pass

t0 = time.time()
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, env=ENV)

peak_rss = 0
while proc.poll() is None:
    try:
        with open(f'/proc/{proc.pid}/status') as f:
            for line in f:
                if line.startswith('VmRSS:'):
                    kb = int(line.split()[1])
                    peak_rss = max(peak_rss, kb)
    except FileNotFoundError:
        break
    time.sleep(2)

stdout, stderr = proc.communicate()
elapsed = time.time() - t0

# Parse output
pieces = [l[6:] for l in stdout.splitlines() if l.startswith('piece:')]
raw_text = ''.join(pieces)

# Strip special tokens
for st in ['<|im_end|>', '<|im_start|>', '<|think|>', '<|/think|>']:
    raw_text = raw_text.replace(st, '')

# Check GPU init
cuda_lines = [l for l in stderr.splitlines() if 'GPU' in l or 'using' in l or 'scheduler' in l]

print(f'\n=== RESULT ===')
for l in cuda_lines:
    print(f'  {l}')
print(f'Tokens: {len(pieces)}')
print(f'Time: {elapsed:.1f}s')
if len(pieces) > 0:
    print(f'Speed: {len(pieces)/elapsed:.2f} t/s')
print(f'Peak RSS: {peak_rss/1024:.0f} MB')
print(f'\nRaw output: {repr(raw_text[:300])}')
print(f'Output: {raw_text[:300]}')

evict_lines = [l for l in stderr.splitlines() if 'expert_router' in l.lower()]
if evict_lines:
    print(f'\n=== ExpertRouter ===')
    for l in evict_lines[:5]:
        print(f'  {l}')

print(f'\n=== RAM AFTER ===')
print(f'  Available: {psutil.virtual_memory().available / 1024**3:.1f} GB')
print(f'  Used: {psutil.virtual_memory().used / 1024**3:.1f} GB')


In [ ]:
#@title Start Lamio HTTP Server (GPU-enabled)
import subprocess, os, time, sys, requests

LAMIO_DIR = '/tmp/lamio'
SERVER_PY = f'{LAMIO_DIR}/v2/server.py'
MODEL_PATH = '/kaggle/working/models/Ornith-1.0-35B-MXFP4_MOE-Only-Imatrix.gguf'

env = dict(os.environ,
    LD_LIBRARY_PATH=f'{LAMIO_DIR}/build/bin',
    LAMIO_MAX_RSS_MB='4096',
    LAMIO_THREADS=str(os.cpu_count()),
    LAMIO_MODEL=MODEL_PATH,
)

subprocess.run(['pkill', '-f', 'server.py'], capture_output=True)
time.sleep(1)

proc = subprocess.Popen(['python3', SERVER_PY], cwd=f'{LAMIO_DIR}/v2', env=env,
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

for i in range(15):
    time.sleep(2)
    try:
        r = requests.get('http://0.0.0.0:5180/api/health', timeout=3)
        if r.status_code == 200:
            print(f'Server online: {r.json()}')
            break
    except: pass
else:
    print('Server failed to start')
    sys.exit(1)

print(f'\nServer running on http://0.0.0.0:5180')
print(f'Model: {os.path.basename(MODEL_PATH)}')
print('\nKeeping kernel alive...')
while True:
    time.sleep(60)
    try:
        r = requests.get('http://0.0.0.0:5180/api/health', timeout=5)
        import psutil
        print(f'  Health: {r.json()["status"]} | RAM: {psutil.virtual_memory().available/1024**3:.1f}GB free')
    except:
        print('  Server may have crashed')
        break


In [ ]:
#@title Chat with model (reuses running server)
import requests, time

URL = 'http://0.0.0.0:5180/v1/chat/completions'

if 'conversation' not in dir():
    conversation = []

# Change this each time you want to chat
user_msg = 'explain why sky is blue in simple words. One phrase.'

conversation.append({'role': 'user', 'content': user_msg})
print(f'>>> User: {user_msg}')

t0 = time.time()
r = requests.post(URL, json={
    'messages': conversation,
    'max_tokens': 256,
    'temperature': 0.3,
    'stream': True,
}, stream=True, timeout=600)

reply = ''
for line in r.iter_lines():
    line = line.decode('utf-8', errors='ignore').strip()
    if line.startswith('data: ') and '[DONE]' not in line:
        import json as _json
        try:
            chunk = _json.loads(line[6:])
            delta = chunk.get('choices', [{}])[0].get('delta', {}).get('content', '')
            if delta: reply += delta
        except: pass

elapsed = time.time() - t0
print(f'<<< Assistant ({elapsed:.1f}s): {reply[:500]}')

conversation.append({'role': 'assistant', 'content': reply})

import subprocess as sp
try:
    out = sp.check_output(['ps', 'aux'], text=True)
    for line in out.splitlines():
        if 'build/src/lamio' in line and 'grep' not in line:
            rss = int(line.split()[5]) // 1024
            print(f'[RSS: {rss} MB]')
            break
except: pass
